In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.integrate import quad

# ----------------------------
# Config
ALPHA = 0.05   # 95% VaR/ES 左尾
COL_NAME = None  # 如果知道列名，填成字符串；若为 None 则使用第一列
# ----------------------------

def expected_shortfall_t(x: np.ndarray, alpha: float = 0.05):
    """
    使用 location-scale Student-t 拟合数据并计算左尾(α)的期望损失 ES。
    返回:
      es_abs: 绝对ES（正数，表示损失幅度）
      es_diff_from_mean: ES相对均值的距离（正数）
      params: (nu, mu, sigma)
      var_level: VaR分位点（作为参考，通常为负值）
    """
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        raise ValueError("Input series is empty after dropping NaNs.")
    if not (0 < alpha < 0.5):
        raise ValueError("alpha should be in (0, 0.5) for left-tail ES.")

    # 拟合 t 分布：返回 (df, loc, scale)
    nu, mu, sigma = stats.t.fit(x)

    # α 分位点（左尾 VaR 水平，通常为负）
    t_alpha = stats.t.ppf(alpha, nu)
    var_level = mu + sigma * t_alpha

    # 数值积分计算 E[X | X <= VaR]
    # 被积函数：x * f_X(x)
    def integrand(z):
        return z * stats.t.pdf((z - mu) / sigma, nu) / sigma

    # 左端积分下限取极小分位，避免 -inf
    lower_bound = mu + sigma * stats.t.ppf(1e-12, nu)

    integral_val, _ = quad(integrand, lower_bound, var_level, limit=200)

    # 条件期望：1/alpha * ∫_{-∞}^{VaR} x f(x) dx
    cond_expectation = integral_val / alpha  # 负数（左尾）

    # 约定：ES 取正数表示损失幅度
    es_abs = -cond_expectation
    es_diff_from_mean = (mu - cond_expectation)

    return es_abs, es_diff_from_mean, (nu, mu, sigma), var_level

def main():
    # Read data
    DATA_DIR = Path.cwd() / "testfiles_" / "data"
    CSV_PATH = DATA_DIR / "test7_2.csv"
    df = pd.read_csv(CSV_PATH, header=0)

    # 取目标列
    if COL_NAME is None:
        series = df.iloc[:, 0].values
        used_col = df.columns[0]
    else:
        series = df[COL_NAME].values
        used_col = COL_NAME

    es_abs, es_diff_mean, params, var_level = expected_shortfall_t(series, alpha=ALPHA)
    nu, mu, sigma = params

    print(f"ES Absolute: {es_abs}")
    print(f"ES Diff from Mean: {es_diff_mean}")

if __name__ == "__main__":
    main()

ES Absolute: 0.07523208715445341
ES Diff from Mean: 0.12117246720180755
